In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

print("python:", sys.executable)

ROOT = Path.cwd().parent
DATA = ROOT / "data" / "raw"
print("project root:", ROOT)
print([p.relative_to(DATA).as_posix() for p in sorted(DATA.glob("*/*.parquet"))])

python: c:\Projects\aml-fraud-detection\.venv\Scripts\python.exe
project root: c:\Projects\aml-fraud-detection
['benchmark/test.parquet', 'benchmark/train.parquet', 'benchmark/valid.parquet', 'chronological/test.parquet', 'chronological/train.parquet', 'chronological/valid.parquet']


In [2]:
%load_ext autoreload
%autoreload 2

sys.path.insert(0, str(ROOT))
from src.data import load_split
from src.features import make_features

valid = load_split("benchmark", "valid")
print("raw:", valid.shape, "| fraud:", int(valid["is_fraud"].sum()))
print("raw RAM (MB):", round(valid.memory_usage(deep=True).sum() / 1e6))

X_valid = make_features(valid)
print("features:", X_valid.shape)
print("feature RAM (MB):", round(X_valid.memory_usage(deep=True).sum() / 1e6))
print(list(X_valid.columns))

raw: (1800359, 15) | fraud: 1311
raw RAM (MB): 70
features: (1800359, 23)
feature RAM (MB): 74
['log_amount', 'log_time_since_last', 'log_velocity', 'spending_deviation_score', 'burst', 'has_history', 'hour_sin', 'hour_cos', 'day_of_week', 'is_first_transaction', 'is_weekend', 'is_cross_bank_transfer', 'is_cross_currency_transfer', 'is_new_receiver', 'is_new_bank', 'is_new_payment_format', 'channel_ACH', 'channel_Credit Card', 'channel_Cheque', 'channel_Bitcoin', 'channel_Reinvestment', 'channel_Wire', 'channel_Cash']


In [3]:
from src.data import load_split
from src.features import make_features, FEATURE_COLUMNS, CHANNELS

train = load_split("benchmark", "train")
X_train = make_features(train)

print("columns in expected order:", list(X_train.columns) == FEATURE_COLUMNS)

chan = X_train[[f"channel_{c}" for c in CHANNELS]]
print("every row has exactly one channel:", bool((chan.sum(axis=1) == 1).all()))
print()
print("rows per channel:")
print(chan.sum().astype(int))
print()
print("fraud total:", int(train["is_fraud"].sum()))
print("fraud in ACH:", int(train.loc[X_train["channel_ACH"] == 1, "is_fraud"].sum()))
print("RAM raw / features (MB):",
      round(train.memory_usage(deep=True).sum() / 1e6),
      "/", round(X_train.memory_usage(deep=True).sum() / 1e6))

columns in expected order: True
every row has exactly one channel: True

rows per channel:
channel_ACH              977407
channel_Bitcoin          318526
channel_Cash             802058
channel_Cheque          3057445
channel_Credit Card     2173776
channel_Reinvestment     791659
channel_Wire             280804
dtype: int64

fraud total: 6119
fraud in ACH: 4954
RAM raw / features (MB): 328 / 344


In [4]:
import gc
from sklearn.metrics import average_precision_score

from src.data import load_split
from src.features import make_features

X, y = {}, {}
for part in ["train", "valid"]:
    raw = load_split("benchmark", part)
    y[part] = raw["is_fraud"].to_numpy()
    X[part] = make_features(raw)
    del raw
    gc.collect()

v = X["valid"]
rule = ((v["channel_ACH"] == 1) & (v["is_new_receiver"] == 1)).to_numpy().astype(int)
hits = int(((rule == 1) & (y["valid"] == 1)).sum())

print(f"flagged: {rule.sum():,} of {len(rule):,}")
print(f"precision: {hits / rule.sum():.4f} | recall: {hits / y['valid'].sum():.4f}")
print(f"PR-AUC of rule: {average_precision_score(y['valid'], rule):.4f}")
print(f"random baseline PR-AUC: {y['valid'].mean():.4f}")

flagged: 56,918 of 1,800,359
precision: 0.0166 | recall: 0.7201
PR-AUC of rule: 0.0121
random baseline PR-AUC: 0.0007


In [5]:
import time
import lightgbm as lgb
from src.evaluate import budget_table

BUDGETS = [500, 1000, 2000, 5000, 10000, 20000, 56918]

t0 = time.time()
lgbm = lgb.LGBMClassifier(
    n_estimators=1500, learning_rate=0.05, num_leaves=15,
    min_child_samples=500, reg_lambda=10.0,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1,
)
lgbm.fit(
    X["train"], y["train"],
    eval_set=[(X["valid"], y["valid"])],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)],
)
p_lgbm = lgbm.predict_proba(X["valid"])[:, 1]

print("minutes:", round((time.time() - t0) / 60, 1))
print("best iteration:", lgbm.best_iteration_)
print(f"PR-AUC (valid): {average_precision_score(y['valid'], p_lgbm):.4f}")
print(budget_table(p_lgbm, y["valid"], BUDGETS).round(4).to_string(index=False))

c:\Projects\aml-fraud-detection\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's binary_logloss: 0.00306414
[200]	valid_0's binary_logloss: 0.00300293
[300]	valid_0's binary_logloss: 0.0029723
[400]	valid_0's binary_logloss: 0.00295453
[500]	valid_0's binary_logloss: 0.00294068
[600]	valid_0's binary_logloss: 0.00292954
[700]	valid_0's binary_logloss: 0.00292555
[800]	valid_0's binary_logloss: 0.00292163
[900]	valid_0's binary_logloss: 0.00291774
[1000]	valid_0's binary_logloss: 0.00291392
[1100]	valid_0's binary_logloss: 0.00291082
[1200]	valid_0's binary_logloss: 0.00290958
[1300]	valid_0's binary_logloss: 0.00290753
[1400]	valid_0's binary_logloss: 0.00290729
[1500]	valid_0's binary_logloss: 0.00290653
Did not meet early stopping. Best iteration is:
[1499]	valid_0's binary_logloss: 0.00290646
minutes: 16.7
best iteration: 1499
PR-AUC (valid): 0.4104
 alerts  precision  recall
    500     0.8480  0.3234
   1000     0.5100  0.3890
   2000     0.2880  0.4394
   5000     0.1340  0.5111
  1

In [7]:
REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)

configs = [
    {"num_leaves": 15, "min_child_samples": 500,  "reg_lambda": 10.0},  # reference (v2 settings)
    {"num_leaves": 31, "min_child_samples": 1000, "reg_lambda": 10.0},
    {"num_leaves": 15, "min_child_samples": 2000, "reg_lambda": 50.0},
    {"num_leaves": 63, "min_child_samples": 500,  "reg_lambda": 50.0},
    {"num_leaves": 31, "min_child_samples": 200,  "reg_lambda": 10.0},
]

rows = []
for i, cfg in enumerate(configs, start=1):
    t0 = time.time()
    m = lgb.LGBMClassifier(
        n_estimators=800, learning_rate=0.1,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbose=-1, **cfg,
    )
    m.fit(X["train"], y["train"],
          eval_set=[(X["valid"], y["valid"])],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    p = m.predict_proba(X["valid"])[:, 1]
    order = np.argsort(-p)
    rows.append({
        **cfg,
        "trees": m.best_iteration_,
        "PR_AUC": average_precision_score(y["valid"], p),
        "prec@500": y["valid"][order[:500]].mean(),
        "prec@5000": y["valid"][order[:5000]].mean(),
        "minutes": (time.time() - t0) / 60,
    })
    pd.DataFrame(rows).to_csv(REPORTS / "tuning_results.csv", index=False)
    print(f"done {i}/{len(configs)} in {rows[-1]['minutes']:.1f} min")

tuning = pd.DataFrame(rows).sort_values("PR_AUC", ascending=False)
print()
print(tuning.round(4).to_string(index=False))

c:\Projects\aml-fraud-detection\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


done 1/5 in 10.4 min


c:\Projects\aml-fraud-detection\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


done 2/5 in 6.6 min


c:\Projects\aml-fraud-detection\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


done 3/5 in 11.8 min


c:\Projects\aml-fraud-detection\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


done 4/5 in 9.0 min


c:\Projects\aml-fraud-detection\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


done 5/5 in 8.2 min

 num_leaves  min_child_samples  reg_lambda  trees  PR_AUC  prec@500  prec@5000  minutes
         31                200        10.0    395  0.4086     0.844     0.1358   8.1565
         15                500        10.0    646  0.4084     0.844     0.1324  10.4405
         63                500        50.0    276  0.4066     0.840     0.1344   9.0141
         31               1000        10.0    317  0.4066     0.836     0.1338   6.5881
         15               2000        50.0    676  0.4045     0.832     0.1344  11.7789


In [2]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

from src.data import load_split
from src.features import make_features

meta = json.loads((ROOT / "models" / "lgbm_v1_meta.json").read_text())
booster = lgb.Booster(model_file=str(ROOT / "models" / meta["model_file"]))

valid = load_split("benchmark", "valid")
y_valid = valid["is_fraud"].to_numpy()
X_valid = make_features(valid)[meta["feature_columns"]]
del valid

p = booster.predict(X_valid)
print("reloaded PR-AUC:", round(average_precision_score(y_valid, p), 4),
      "| saved in meta:", meta["valid_pr_auc"])
print("mean predicted:", round(p.mean(), 5), "| actual fraud rate:", round(y_valid.mean(), 5))

bins = [0, 0.0005, 0.001, 0.005, 0.02, 0.1, 0.3, 0.6, 1.0]
cal = pd.DataFrame({"p": p, "y": y_valid})
cal["bin"] = pd.cut(cal["p"], bins=bins, include_lowest=True)
table = cal.groupby("bin", observed=True).agg(
    rows=("y", "size"), fraud=("y", "sum"),
    mean_pred=("p", "mean"), actual_rate=("y", "mean"),
)
print(table.round(5).to_string())

reloaded PR-AUC: 0.4104 | saved in meta: 0.4104
mean predicted: 0.00074 | actual fraud rate: 0.00073
                     rows  fraud  mean_pred  actual_rate
bin                                                     
(-0.001, 0.0005]  1564454     16    0.00002      0.00001
(0.0005, 0.001]     56310     61    0.00078      0.00108
(0.001, 0.005]     145889    253    0.00184      0.00173
(0.005, 0.02]       26594    268    0.01010      0.01008
(0.02, 0.1]          6075    197    0.03775      0.03243
(0.1, 0.3]            499     83    0.16669      0.16633
(0.3, 0.6]            134     55    0.43283      0.41045
(0.6, 1.0]            404    378    0.88962      0.93564
